# AIML Lab Assignment #9 — Autoencoder for Feature Learning
**Course:** CSET301 | **Dataset:** MNIST (Image) + Wine (Tabular) | **Semester:** 4th Even 2025

---

## What is an Autoencoder?

An **Autoencoder** is a type of neural network that learns to:
1. **Compress** (encode) input data into a small representation called the **latent space** or **bottleneck**
2. **Reconstruct** (decode) the original input from that small representation

```
Input (784 pixels)
      ↓
  ENCODER          ← compresses data
      ↓
Latent Space (32)  ← tiny compressed version
      ↓
  DECODER          ← reconstructs data
      ↓
Output (784 pixels) ← should look like input!
```

### Why is this useful?
- The latent space captures the **most important features** of the data
- These features can be used for **classification, denoising, or compression**
- The model learns WITHOUT labels — it is **unsupervised learning**

### What we will build:
1. **Basic Autoencoder (AE)** — Standard encoder-decoder
2. **Denoising Autoencoder (DAE)** — Learns to remove noise from images
3. **Sparse Autoencoder (SAE)** — Forces most latent neurons to stay inactive

---
### Analogy
> Think of it like zipping a file. The encoder = zip compression. The latent space = the zip file. The decoder = unzipping. A good autoencoder compresses WITHOUT losing important information.

---
## Step 0: Import All Libraries

In [ ]:
# ================================================================
# IMPORT ALL LIBRARIES
# ================================================================

# numpy → work with arrays and numbers
import numpy as np

# pandas → work with tables
import pandas as pd

# matplotlib & seaborn → draw graphs and visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# tensorflow & keras → build and train autoencoders
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

# sklearn → datasets, splits, metrics, downstream classifiers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# suppress warnings
import warnings
warnings.filterwarnings('ignore')

# Fix random seeds so results are reproducible every run
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("All libraries imported successfully!")

---
## Task 1: Data Exploration

We use **two datasets**:
- **MNIST** — 28×28 grayscale images of handwritten digits (image dataset)
- **Wine** — 13 chemical measurements of wines from 3 grape varieties (tabular dataset)

### 1A — MNIST Image Dataset

In [ ]:
# ================================================================
# 1A.1  LOAD MNIST DATASET
# ================================================================
# Keras gives us MNIST directly — no manual download needed

(X_mnist_train_raw, y_mnist_train), (X_mnist_test_raw, y_mnist_test) = \
    keras.datasets.mnist.load_data()

print("===== MNIST Dataset Info =====")
print(f"Training images : {X_mnist_train_raw.shape}")
print(f"  → 60,000 images, each 28×28 pixels")
print(f"Test images     : {X_mnist_test_raw.shape}")
print(f"Pixel dtype     : {X_mnist_train_raw.dtype}")
print(f"Pixel range     : {X_mnist_train_raw.min()} to {X_mnist_train_raw.max()}")
print(f"Classes         : {np.unique(y_mnist_train)} (digits 0-9)")
print()
# Key statistics across all pixels
flat_pixels = X_mnist_train_raw.flatten().astype('float32')
print(f"Mean pixel value  : {flat_pixels.mean():.2f}")
print(f"Std  pixel value  : {flat_pixels.std():.2f}")
print(f"Median pixel value: {np.median(flat_pixels):.2f}")

In [ ]:
# ================================================================
# 1A.2  VISUALIZE SAMPLE MNIST IMAGES
# ================================================================

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
fig.suptitle('MNIST Sample Images — 2 per digit class', fontsize=13, fontweight='bold')

for digit in range(10):
    idx1 = np.where(y_mnist_train == digit)[0][0]
    idx2 = np.where(y_mnist_train == digit)[0][1]
    axes[0, digit].imshow(X_mnist_train_raw[idx1], cmap='gray')
    axes[0, digit].set_title(f'{digit}', fontsize=10)
    axes[0, digit].axis('off')
    axes[1, digit].imshow(X_mnist_train_raw[idx2], cmap='gray')
    axes[1, digit].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 1A.3  CLASS DISTRIBUTION — is dataset balanced?
# ================================================================

classes, counts = np.unique(y_mnist_train, return_counts=True)
plt.figure(figsize=(10, 4))
plt.bar(classes, counts, color='steelblue', edgecolor='black', width=0.6)
plt.title('MNIST Class Distribution (Training Set)', fontweight='bold')
plt.xlabel('Digit Class')
plt.ylabel('Number of Images')
plt.xticks(classes)
for c, cnt in zip(classes, counts):
    plt.text(c, cnt + 80, str(cnt), ha='center', fontsize=8)
plt.tight_layout()
plt.show()
print("Dataset is balanced — roughly 6000 images per digit class.")

### 1B — Wine Tabular Dataset

In [ ]:
# ================================================================
# 1B.1  LOAD WINE DATASET
# ================================================================
# Wine dataset: 178 wine samples, 13 chemical measurements
# Classes: 3 types of wine (from 3 different grape cultivars)

wine_data = load_wine()
df_wine = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)
df_wine['target'] = wine_data.target  # 0, 1, or 2

print("===== Wine Dataset Info =====")
print(f"Shape  : {df_wine.shape}  → {df_wine.shape[0]} rows × {df_wine.shape[1]} columns")
print(f"Classes: {wine_data.target_names}")
print(f"Samples per class: {np.bincount(wine_data.target)}")
print()
print("Feature statistics:")
print(df_wine.drop('target', axis=1).describe().round(2))

In [ ]:
# ================================================================
# 1B.2  CHECK FOR MISSING VALUES
# ================================================================

missing_mnist = 0  # MNIST is always clean
missing_wine  = df_wine.isnull().sum().sum()

print("Missing values in MNIST  :", missing_mnist)
print("Missing values in Wine   :", missing_wine)
print()
if missing_wine == 0:
    print("Both datasets are clean — no missing values!")
else:
    print("Wine has missing values — handling with median imputation")
    df_wine = df_wine.fillna(df_wine.median())

In [ ]:
# ================================================================
# 1B.3  VISUALIZE WINE FEATURES — histograms
# ================================================================

feature_cols = wine_data.feature_names
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
fig.suptitle('Wine Dataset — Feature Distributions', fontsize=13, fontweight='bold')

for idx, (feat, ax) in enumerate(zip(feature_cols, axes.flat)):
    ax.hist(df_wine[feat], bins=20, color='coral', edgecolor='black', alpha=0.7)
    ax.set_title(feat, fontsize=8)
    ax.set_xlabel('Value', fontsize=7)
    ax.set_ylabel('Count', fontsize=7)

# Hide unused subplots
for ax in axes.flat[len(feature_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# 1B.4  CORRELATION HEATMAP — which features are related?
# ================================================================
# Correlation measures how much two features move together
# +1 = perfectly positively correlated, -1 = perfectly inversely correlated

plt.figure(figsize=(13, 10))
corr_matrix = df_wine[feature_cols].corr()
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, linewidths=0.5, annot_kws={'size': 7}
)
plt.title('Wine Feature Correlation Heatmap\n(Red = positive, Blue = negative)', fontweight='bold')
plt.tight_layout()
plt.show()
print("High correlation between features helps the autoencoder compress efficiently.")

---
## Task 2: Data Preparation
We prepare both datasets in the format needed for autoencoders.

In [ ]:
# ================================================================
# 2A  PREPARE MNIST DATA
# ================================================================

# Step 1: Normalize pixels from 0-255 → 0.0-1.0
X_mnist_all_raw = np.concatenate([X_mnist_train_raw, X_mnist_test_raw], axis=0)
y_mnist_all     = np.concatenate([y_mnist_train, y_mnist_test], axis=0)

X_mnist_norm = X_mnist_all_raw.astype('float32') / 255.0

# Step 2: Flatten images from (70000, 28, 28) → (70000, 784)
# MLP autoencoder needs flat 1D input — 28×28 = 784 pixels
X_mnist_flat = X_mnist_norm.reshape(-1, 784)

# Step 3: Split into Train 60% / Val 20% / Test 20%
X_m_trainval, X_m_test, y_m_trainval, y_m_test = train_test_split(
    X_mnist_flat, y_mnist_all,
    test_size=0.20, random_state=42, stratify=y_mnist_all
)
X_m_train, X_m_val, y_m_train, y_m_val = train_test_split(
    X_m_trainval, y_m_trainval,
    test_size=0.25, random_state=42, stratify=y_m_trainval
)

print("===== MNIST Split Summary =====")
print(f"Train : {X_m_train.shape[0]:>6} images  (60%)")
print(f"Val   : {X_m_val.shape[0]:>6} images  (20%)")
print(f"Test  : {X_m_test.shape[0]:>6} images  (20%)")
print(f"Input size per image: {X_m_train.shape[1]} (flattened 28×28)")
print(f"Pixel range after normalization: {X_m_train.min():.1f} to {X_m_train.max():.1f}")

In [ ]:
# ================================================================
# 2B  PREPARE WINE DATA
# ================================================================

# Step 1: Separate features (X) and labels (y)
X_wine_raw = df_wine[feature_cols].values   # shape: (178, 13)
y_wine     = df_wine['target'].values       # shape: (178,)

# Step 2: Standardize features — mean=0, std=1
# IMPORTANT: Fit scaler ONLY on training data, then transform all sets
X_w_trainval, X_w_test, y_w_trainval, y_w_test = train_test_split(
    X_wine_raw, y_wine, test_size=0.20, random_state=42, stratify=y_wine
)
X_w_train, X_w_val, y_w_train, y_w_val = train_test_split(
    X_w_trainval, y_w_trainval, test_size=0.25, random_state=42, stratify=y_w_trainval
)

scaler = StandardScaler()
X_w_train = scaler.fit_transform(X_w_train)  # fit+transform on train
X_w_val   = scaler.transform(X_w_val)        # only transform on val
X_w_test  = scaler.transform(X_w_test)       # only transform on test

print("===== Wine Split Summary =====")
print(f"Train : {X_w_train.shape[0]:>3} samples  (60%)")
print(f"Val   : {X_w_val.shape[0]:>3} samples  (20%)")
print(f"Test  : {X_w_test.shape[0]:>3} samples  (20%)")
print(f"Features per sample: {X_w_train.shape[1]}")
print(f"After standardization — mean: {X_w_train.mean():.4f}, std: {X_w_train.std():.4f}")

---
## Task 3: Model Implementation

### 3A — Basic Autoencoder (AE)

**Architecture:**
```
Encoder:  784 → 256 → 128 → 32  (latent)
Decoder:   32 → 128 → 256 → 784 (reconstruction)
```
The model learns to **compress 784 pixels into 32 numbers**, then reconstruct back.
Loss = **MSE** (Mean Squared Error between original and reconstructed pixels)

In [ ]:
# ================================================================
# 3A.1  BUILD BASIC AUTOENCODER
# ================================================================

LATENT_DIM   = 32    # size of the bottleneck (compressed representation)
INPUT_DIM    = 784   # number of input features (28×28 pixels)

def build_basic_autoencoder(input_dim=784, latent_dim=32):
    """
    Basic MLP Autoencoder.
    
    ENCODER compresses: input_dim → 256 → 128 → latent_dim
    DECODER expands:    latent_dim → 128 → 256 → input_dim
    """
    # ---- ENCODER ----
    # Takes input and compresses it step by step
    encoder_input = keras.Input(shape=(input_dim,), name='encoder_input')

    # Gradually reduce dimension: 784 → 256
    x = layers.Dense(256, activation='relu', name='enc_dense_1')(encoder_input)
    # 256 → 128
    x = layers.Dense(128, activation='relu', name='enc_dense_2')(x)
    # 128 → 32 (the bottleneck / latent space)
    # No activation on latent layer — keep raw values
    latent = layers.Dense(latent_dim, activation='relu', name='latent')(x)

    # Create the encoder model (we will use this separately to extract features)
    encoder = keras.Model(encoder_input, latent, name='Encoder')

    # ---- DECODER ----
    # Takes the compressed latent vector and rebuilds the original
    decoder_input = keras.Input(shape=(latent_dim,), name='decoder_input')

    # Mirror of encoder: 32 → 128
    x = layers.Dense(128, activation='relu', name='dec_dense_1')(decoder_input)
    # 128 → 256
    x = layers.Dense(256, activation='relu', name='dec_dense_2')(x)
    # 256 → 784 (reconstruct original image)
    # sigmoid activation: keeps output in [0,1] range (matching normalized pixels)
    reconstructed = layers.Dense(input_dim, activation='sigmoid', name='output')(x)

    # Create the decoder model
    decoder = keras.Model(decoder_input, reconstructed, name='Decoder')

    # ---- FULL AUTOENCODER ----
    # Connect encoder + decoder into one model
    ae_output = decoder(encoder(encoder_input))
    autoencoder = keras.Model(encoder_input, ae_output, name='Basic_Autoencoder')

    return autoencoder, encoder, decoder


# Build the model
ae_basic, encoder_basic, decoder_basic = build_basic_autoencoder(
    input_dim=INPUT_DIM,
    latent_dim=LATENT_DIM
)

# Show model structure
ae_basic.summary()
print()
print(f"Compression ratio: {INPUT_DIM} → {LATENT_DIM} ({INPUT_DIM/LATENT_DIM:.1f}x compression!)")

In [ ]:
# ================================================================
# 3A.2  COMPILE AND TRAIN BASIC AUTOENCODER
# ================================================================
# NOTE: For autoencoders, input = output (X_train, X_train)
# The model tries to reconstruct its own input!

ae_basic.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse'  # Mean Squared Error: measures how different reconstruction is from original
)

early_stop_ae = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print("Training Basic Autoencoder...")
print("Input = Output = X_train (model learns to reconstruct itself)")
print()

history_ae = ae_basic.fit(
    X_m_train, X_m_train,      # ← input AND target are both X_train!
    epochs=30,
    batch_size=64,
    validation_data=(X_m_val, X_m_val),  # ← same for val
    callbacks=[early_stop_ae],
    verbose=1
)

# Evaluate on test set
ae_test_loss = ae_basic.evaluate(X_m_test, X_m_test, verbose=0)
print(f"\nBasic AE — Test MSE Loss: {ae_test_loss:.6f}")
print("Lower MSE = better reconstruction")

In [ ]:
# ================================================================
# 3A.3  VISUALIZE RECONSTRUCTIONS — Original vs Reconstructed
# ================================================================
# This shows HOW WELL the autoencoder reconstructed the images

# Get 10 test images and reconstruct them
sample_imgs    = X_m_test[:10]
reconstructed  = ae_basic.predict(sample_imgs, verbose=0)

fig, axes = plt.subplots(2, 10, figsize=(20, 4))
fig.suptitle('Basic Autoencoder — Original (top) vs Reconstructed (bottom)',
             fontsize=13, fontweight='bold')

for i in range(10):
    # Top row: original images
    axes[0, i].imshow(sample_imgs[i].reshape(28, 28), cmap='gray')
    axes[0, i].set_title('Original', fontsize=7)
    axes[0, i].axis('off')

    # Bottom row: reconstructed images
    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray')
    # Calculate MSE for this single image
    mse_val = np.mean((sample_imgs[i] - reconstructed[i])**2)
    axes[1, i].set_title(f'MSE:{mse_val:.4f}', fontsize=7)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()
print("The reconstructed images should look like blurry versions of the originals.")
print("Lower MSE = better reconstruction quality for that image.")

---
### 3B — Denoising Autoencoder (DAE)

A **Denoising Autoencoder** is trained on **noisy inputs** but learns to output **clean images**.

```
Noisy Image → Encoder → Latent → Decoder → Clean Image
```

This forces the encoder to learn the **underlying structure** of the data, not just memorize noise.

In [ ]:
# ================================================================
# 3B.1  ADD NOISE TO IMAGES
# ================================================================

def add_gaussian_noise(images, noise_factor=0.3):
    """
    Add Gaussian (random) noise to images.
    noise_factor controls how much noise to add.
    np.clip keeps pixel values in [0, 1] range.
    """
    # Generate random noise — same shape as images, values from normal distribution
    noise = np.random.normal(loc=0.0, scale=noise_factor, size=images.shape)
    # Add noise to images, then clip to keep in valid [0, 1] range
    noisy_images = np.clip(images + noise, 0.0, 1.0)
    return noisy_images.astype('float32')


NOISE_FACTOR = 0.3   # 0=no noise, 1=maximum noise

# Create noisy versions of each split
X_m_train_noisy = add_gaussian_noise(X_m_train, NOISE_FACTOR)
X_m_val_noisy   = add_gaussian_noise(X_m_val,   NOISE_FACTOR)
X_m_test_noisy  = add_gaussian_noise(X_m_test,  NOISE_FACTOR)

# Show comparison: clean vs noisy
fig, axes = plt.subplots(2, 8, figsize=(18, 4))
fig.suptitle(f'Adding Gaussian Noise (factor={NOISE_FACTOR}) to MNIST Images',
             fontsize=12, fontweight='bold')
for i in range(8):
    axes[0, i].imshow(X_m_train[i].reshape(28, 28), cmap='gray')
    axes[0, i].set_title('Clean', fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(X_m_train_noisy[i].reshape(28, 28), cmap='gray')
    axes[1, i].set_title('Noisy', fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()
print("Top row = clean images. Bottom row = same images with noise added.")
print("The DAE will learn to remove this noise!")

In [ ]:
# ================================================================
# 3B.2  BUILD AND TRAIN DENOISING AUTOENCODER
# ================================================================
# Same architecture as Basic AE
# BUT: input = noisy images, target = clean images

ae_dae, encoder_dae, decoder_dae = build_basic_autoencoder(
    input_dim=INPUT_DIM,
    latent_dim=LATENT_DIM
)
ae_dae._name = 'Denoising_Autoencoder'

ae_dae.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse'
)

early_stop_dae = EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)

print("Training Denoising Autoencoder...")
print("Input = NOISY images, Target = CLEAN images")
print("The model learns to remove noise!")
print()

history_dae = ae_dae.fit(
    X_m_train_noisy, X_m_train,         # noisy input → clean target
    epochs=30,
    batch_size=64,
    validation_data=(X_m_val_noisy, X_m_val),  # same for val
    callbacks=[early_stop_dae],
    verbose=1
)

dae_test_loss = ae_dae.evaluate(X_m_test_noisy, X_m_test, verbose=0)
print(f"\nDenoising AE — Test MSE Loss: {dae_test_loss:.6f}")

In [ ]:
# ================================================================
# 3B.3  VISUALIZE DAE RESULTS — Noisy vs Denoised
# ================================================================

sample_noisy   = X_m_test_noisy[:10]
sample_clean   = X_m_test[:10]
denoised_imgs  = ae_dae.predict(sample_noisy, verbose=0)

fig, axes = plt.subplots(3, 10, figsize=(20, 6))
fig.suptitle('Denoising Autoencoder Results\n'
             'Row 1: Original Clean | Row 2: Noisy Input | Row 3: Denoised Output',
             fontsize=12, fontweight='bold')

for i in range(10):
    axes[0, i].imshow(sample_clean[i].reshape(28, 28), cmap='gray')
    axes[0, i].set_title('Clean', fontsize=7)
    axes[0, i].axis('off')

    axes[1, i].imshow(sample_noisy[i].reshape(28, 28), cmap='gray')
    axes[1, i].set_title('Noisy', fontsize=7)
    axes[1, i].axis('off')

    axes[2, i].imshow(denoised_imgs[i].reshape(28, 28), cmap='gray')
    axes[2, i].set_title('Denoised', fontsize=7)
    axes[2, i].axis('off')

plt.tight_layout()
plt.show()
print("The denoised images (row 3) should look much cleaner than noisy (row 2)!")

---
### 3C — Sparse Autoencoder (SAE)

A **Sparse Autoencoder** adds a **penalty** to keep most latent neurons inactive (close to 0).
Only a few neurons fire for each input — this forces more **meaningful, distributed features**.

We use **L1 regularization** on the latent layer = penalizes large latent values.

In [ ]:
# ================================================================
# 3C.1  BUILD AND TRAIN SPARSE AUTOENCODER
# ================================================================

SPARSITY_LAMBDA = 1e-5   # L1 penalty strength — higher = more sparse

def build_sparse_autoencoder(input_dim=784, latent_dim=32, l1_lambda=1e-5):
    """
    Sparse Autoencoder — same structure as basic AE
    but adds L1 regularization on the latent layer.
    L1 penalty pushes most latent values toward 0 (sparse).
    """
    encoder_input = keras.Input(shape=(input_dim,))
    x = layers.Dense(256, activation='relu')(encoder_input)
    x = layers.Dense(128, activation='relu')(x)
    # activity_regularizer adds L1 penalty to the OUTPUT of this layer
    # This forces most latent neuron activations to stay near 0
    latent = layers.Dense(
        latent_dim, activation='relu',
        activity_regularizer=regularizers.l1(l1_lambda),  # ← sparsity penalty
        name='sparse_latent'
    )(x)

    encoder = keras.Model(encoder_input, latent, name='Sparse_Encoder')

    decoder_input = keras.Input(shape=(latent_dim,))
    x = layers.Dense(128, activation='relu')(decoder_input)
    x = layers.Dense(256, activation='relu')(x)
    reconstructed = layers.Dense(input_dim, activation='sigmoid')(x)

    decoder = keras.Model(decoder_input, reconstructed, name='Sparse_Decoder')

    ae_output = decoder(encoder(encoder_input))
    autoencoder = keras.Model(encoder_input, ae_output, name='Sparse_Autoencoder')

    return autoencoder, encoder, decoder


ae_sae, encoder_sae, decoder_sae = build_sparse_autoencoder(
    input_dim=INPUT_DIM, latent_dim=LATENT_DIM, l1_lambda=SPARSITY_LAMBDA
)

ae_sae.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse'
)

early_stop_sae = EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
)

print("Training Sparse Autoencoder...")
print(f"L1 sparsity penalty: {SPARSITY_LAMBDA}")
print("Most latent neurons will be forced close to 0 (sparse representation)")
print()

history_sae = ae_sae.fit(
    X_m_train, X_m_train,
    epochs=30,
    batch_size=64,
    validation_data=(X_m_val, X_m_val),
    callbacks=[early_stop_sae],
    verbose=1
)

sae_test_loss = ae_sae.evaluate(X_m_test, X_m_test, verbose=0)
print(f"\nSparse AE — Test MSE Loss: {sae_test_loss:.6f}")

---
## Task 4: Hyperparameter Tuning
We test different **latent dimensions** to find the best compression level.

In [ ]:
# ================================================================
# TASK 4: LATENT DIMENSION TUNING
# ================================================================
# How much can we compress without losing quality?
# We test latent sizes: 8, 16, 32, 64

latent_dims   = [8, 16, 32, 64]
latent_results = []

for ldim in latent_dims:
    print(f"\nTraining with latent_dim = {ldim} ...")

    m, enc, dec = build_basic_autoencoder(input_dim=INPUT_DIM, latent_dim=ldim)
    m.compile(optimizer='adam', loss='mse')

    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    h = m.fit(
        X_m_train, X_m_train,
        epochs=15, batch_size=64,
        validation_data=(X_m_val, X_m_val),
        callbacks=[es], verbose=0
    )
    test_mse = m.evaluate(X_m_test, X_m_test, verbose=0)
    compression = INPUT_DIM / ldim

    latent_results.append({
        'latent_dim'  : ldim,
        'test_mse'    : test_mse,
        'compression' : compression,
        'history'     : h
    })
    print(f"  Latent={ldim:>3} | Test MSE={test_mse:.5f} | Compression={compression:.1f}x")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Latent Dimension Tuning', fontweight='bold')

# Bar chart: MSE per latent size
mse_vals  = [r['test_mse'] for r in latent_results]
ldim_vals = [r['latent_dim'] for r in latent_results]
axes[0].bar([str(d) for d in ldim_vals], mse_vals, color='steelblue', edgecolor='black')
axes[0].set_title('Test MSE vs Latent Dimension\n(Lower = Better Reconstruction)', fontweight='bold')
axes[0].set_xlabel('Latent Dimension')
axes[0].set_ylabel('Test MSE')
for i, v in enumerate(mse_vals):
    axes[0].text(i, v + 0.0001, f'{v:.5f}', ha='center', fontsize=9)

# Line chart: Validation loss curves
colors = ['blue', 'green', 'red', 'purple']
for r, col in zip(latent_results, colors):
    ep = range(1, len(r['history'].history['val_loss']) + 1)
    axes[1].plot(ep, r['history'].history['val_loss'],
                 color=col, linewidth=2, label=f'latent={r["latent_dim"]}')
axes[1].set_title('Validation Loss Curves per Latent Size', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Val MSE Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("\nKey insight: Larger latent dim = lower MSE (better quality) but less compression.")
print("Trade-off: compression vs reconstruction quality.")

---
## Task 5: Evaluation & Feature Utility
### 5.1 — Reconstruction Quality Comparison

In [ ]:
# ================================================================
# 5.1  COMPARE RECONSTRUCTION QUALITY — AE vs DAE vs SAE
# ================================================================

ae_mse  = ae_basic.evaluate(X_m_test, X_m_test, verbose=0)
dae_mse = ae_dae.evaluate(X_m_test_noisy, X_m_test, verbose=0)
sae_mse = ae_sae.evaluate(X_m_test, X_m_test, verbose=0)

model_names = ['Basic AE', 'Denoising AE', 'Sparse AE']
mse_values  = [ae_mse, dae_mse, sae_mse]

print("===== Reconstruction Quality (Test MSE) =====")
for name, mse in zip(model_names, mse_values):
    print(f"  {name:<15} : {mse:.6f}")

plt.figure(figsize=(8, 4))
colors_bar = ['steelblue', 'coral', 'green']
bars = plt.bar(model_names, mse_values, color=colors_bar, edgecolor='black', width=0.5)
plt.title('Test MSE per Autoencoder Type\n(Lower = Better Reconstruction)', fontweight='bold')
plt.ylabel('Mean Squared Error (MSE)')
for bar, val in zip(bars, mse_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0001,
             f'{val:.5f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 5.2 — Feature Utility (Downstream Classification)
We use the **latent features** extracted by each autoencoder to train a classifier.
Better features → higher classification accuracy.

In [ ]:
# ================================================================
# 5.2A  EXTRACT LATENT FEATURES from each autoencoder
# ================================================================
# The encoder maps images → latent vectors
# These latent vectors are the 'features' we extracted

# Basic AE features
latent_train_ae  = encoder_basic.predict(X_m_train, verbose=0)  # shape: (N, 32)
latent_test_ae   = encoder_basic.predict(X_m_test,  verbose=0)

# Denoising AE features
latent_train_dae = encoder_dae.predict(X_m_train, verbose=0)
latent_test_dae  = encoder_dae.predict(X_m_test,  verbose=0)

# Sparse AE features
latent_train_sae = encoder_sae.predict(X_m_train, verbose=0)
latent_test_sae  = encoder_sae.predict(X_m_test,  verbose=0)

print("Latent features extracted!")
print(f"Original feature size : {X_m_train.shape[1]} (raw pixels)")
print(f"Latent  feature size  : {latent_train_ae.shape[1]} (compressed)")
print()
print(f"AE   latent — mean: {latent_train_ae.mean():.4f}, std: {latent_train_ae.std():.4f}")
print(f"DAE  latent — mean: {latent_train_dae.mean():.4f}, std: {latent_train_dae.std():.4f}")
print(f"SAE  latent — mean: {latent_train_sae.mean():.4f}, std: {latent_train_sae.std():.4f}")
print("\nSAE should have a lower mean (sparse = most neurons close to 0)")

In [ ]:
# ================================================================
# 5.2B  DOWNSTREAM CLASSIFICATION — Logistic Regression
# ================================================================
# Train a simple Logistic Regression on:
#   1. Raw pixel features (784 dims) — baseline
#   2. AE latent features  (32 dims)
#   3. DAE latent features (32 dims)
#   4. SAE latent features (32 dims)
# Compare which feature set gives best classification accuracy!

def evaluate_classifier(X_train_feat, y_train, X_test_feat, y_test, feat_name):
    """Train Logistic Regression and print metrics."""
    clf = LogisticRegression(max_iter=500, random_state=42)
    clf.fit(X_train_feat, y_train)
    y_pred = clf.predict(X_test_feat)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='macro')
    rec  = recall_score(y_test, y_pred, average='macro')
    f1   = f1_score(y_test, y_pred, average='macro')

    print(f"  {feat_name:<25} Acc={acc*100:.2f}%  Prec={prec:.3f}  Rec={rec:.3f}  F1={f1:.3f}")
    return {'name': feat_name, 'accuracy': acc, 'f1': f1}


print("===== Downstream Classification (Logistic Regression) =====")
print(f"  {'Feature Set':<25} {'Accuracy':>10}  {'Precision':>10}  {'Recall':>8}  {'F1':>6}")
print("-" * 70)

# Use a smaller subset of train/test for speed
N_TRAIN = min(5000, len(X_m_train))
N_TEST  = min(1000, len(X_m_test))

downstream_results = []

# 1. Raw pixels (baseline)
r = evaluate_classifier(X_m_train[:N_TRAIN], y_m_train[:N_TRAIN],
                         X_m_test[:N_TEST],  y_m_test[:N_TEST],
                         'Raw Pixels (784)')
downstream_results.append(r)

# 2. AE latent features
r = evaluate_classifier(latent_train_ae[:N_TRAIN], y_m_train[:N_TRAIN],
                         latent_test_ae[:N_TEST],  y_m_test[:N_TEST],
                         'AE Latent (32)')
downstream_results.append(r)

# 3. DAE latent features
r = evaluate_classifier(latent_train_dae[:N_TRAIN], y_m_train[:N_TRAIN],
                         latent_test_dae[:N_TEST],   y_m_test[:N_TEST],
                         'DAE Latent (32)')
downstream_results.append(r)

# 4. SAE latent features
r = evaluate_classifier(latent_train_sae[:N_TRAIN], y_m_train[:N_TRAIN],
                         latent_test_sae[:N_TEST],   y_m_test[:N_TEST],
                         'SAE Latent (32)')
downstream_results.append(r)

print()
print("Note: 32 latent features vs 784 raw pixels — compressed features can match or beat raw!")

In [ ]:
# ================================================================
# 5.2C  WINE DATASET — Downstream Classification
# ================================================================
# Now we apply autoencoders to the tabular Wine dataset

WINE_INPUT_DIM  = X_w_train.shape[1]  # 13 features
WINE_LATENT_DIM = 6                   # compress 13 → 6

ae_wine, enc_wine, dec_wine = build_basic_autoencoder(
    input_dim=WINE_INPUT_DIM,
    latent_dim=WINE_LATENT_DIM
)
ae_wine.compile(optimizer='adam', loss='mse')

print(f"Training Wine Autoencoder (13 → {WINE_LATENT_DIM} features)...")
ae_wine.fit(
    X_w_train, X_w_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_w_val, X_w_val),
    callbacks=[EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    verbose=0
)

# Extract latent features
latent_wine_train = enc_wine.predict(X_w_train, verbose=0)
latent_wine_test  = enc_wine.predict(X_w_test,  verbose=0)

# Compare raw vs latent features
print("\n===== Wine Downstream Classification =====")
print(f"  {'Feature Set':<25} Accuracy")
print("-" * 38)

for feat_name, X_tr, X_te in [
    ('Raw Wine (13 features)', X_w_train, X_w_test),
    ('AE Latent (6 features)', latent_wine_train, latent_wine_test)
]:
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_tr, y_w_train)
    acc = accuracy_score(y_w_test, clf.predict(X_te))
    print(f"  {feat_name:<25} {acc*100:.2f}%")

---
## Task 6: Visualizations
### 6.1 — Training Curves for All Three Autoencoders

In [ ]:
# ================================================================
# 6.1  TRAINING CURVES — AE, DAE, SAE side by side
# ================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training vs Validation Loss — All Three Autoencoders',
             fontsize=13, fontweight='bold')

plot_configs = [
    (history_ae,  'Basic AE',       axes[0]),
    (history_dae, 'Denoising AE',   axes[1]),
    (history_sae, 'Sparse AE',      axes[2]),
]

for hist, title, ax in plot_configs:
    epochs = range(1, len(hist.history['loss']) + 1)
    ax.plot(epochs, hist.history['loss'],     'b-o', markersize=3, label='Train Loss')
    ax.plot(epochs, hist.history['val_loss'], 'r-o', markersize=3, label='Val Loss')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("Both train and val loss should decrease together. Big gap = overfitting.")

### 6.2 — Latent Space Visualization with PCA and t-SNE

In [ ]:
# ================================================================
# 6.2  LATENT SPACE VISUALIZATION — PCA + t-SNE
# ================================================================
# We project the 32D latent vectors down to 2D using PCA and t-SNE
# Then color points by digit class to see if digits cluster together
# Well-separated clusters = the autoencoder learned meaningful features!

# Use a subset for speed
N_VIZ = 2000
lat_viz = latent_test_ae[:N_VIZ]
y_viz   = y_m_test[:N_VIZ]

# ---------- PCA — fast, linear projection ----------
pca_2d = PCA(n_components=2, random_state=42)
lat_pca = pca_2d.fit_transform(lat_viz)

# ---------- t-SNE — slower, non-linear, better clusters ----------
print("Running t-SNE (this takes ~30 seconds)...")
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=500)
lat_tsne = tsne_2d.fit_transform(lat_viz)
print("t-SNE done!")

# Plot both side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Latent Space Visualization — AE Features\n'
             'Each dot = one test image. Color = digit class.',
             fontsize=12, fontweight='bold')

cmap = plt.cm.get_cmap('tab10', 10)

for ax, coords, method in [
    (axes[0], lat_pca,  'PCA (linear)'),
    (axes[1], lat_tsne, 't-SNE (non-linear)')
]:
    sc = ax.scatter(coords[:, 0], coords[:, 1],
                    c=y_viz, cmap=cmap, s=8, alpha=0.7, vmin=0, vmax=9)
    ax.set_title(method, fontweight='bold')
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    plt.colorbar(sc, ax=ax, label='Digit Class', ticks=range(10))
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()
print("Well-separated clusters of same color = autoencoder learned meaningful representations!")
print("t-SNE usually shows cleaner clusters than PCA.")

### 6.3 — Comparison Reconstruction Grid: AE vs DAE vs SAE

In [ ]:
# ================================================================
# 6.3  RECONSTRUCTION GRID — All 3 models compared
# ================================================================

n_samples = 8
sample_orig    = X_m_test[:n_samples]
recon_ae       = ae_basic.predict(sample_orig, verbose=0)
recon_dae      = ae_dae.predict(sample_orig, verbose=0)  # clean input
recon_sae      = ae_sae.predict(sample_orig, verbose=0)

fig, axes = plt.subplots(4, n_samples, figsize=(20, 8))
fig.suptitle('Reconstruction Comparison — 4 rows: Original / AE / DAE / SAE',
             fontsize=12, fontweight='bold')

row_labels = ['Original', 'Basic AE', 'Denoising AE', 'Sparse AE']
all_rows   = [sample_orig, recon_ae, recon_dae, recon_sae]

for row_idx, (row_data, row_label) in enumerate(zip(all_rows, row_labels)):
    for col_idx in range(n_samples):
        ax = axes[row_idx, col_idx]
        ax.imshow(row_data[col_idx].reshape(28, 28), cmap='gray')
        if col_idx == 0:
            ax.set_ylabel(row_label, fontsize=9, rotation=90, labelpad=4)
        ax.axis('off')

plt.tight_layout()
plt.show()

### 6.4 — Downstream Performance Bar Chart

In [ ]:
# ================================================================
# 6.4  DOWNSTREAM PERFORMANCE COMPARISON CHART
# ================================================================

names    = [r['name'] for r in downstream_results]
accs     = [r['accuracy'] * 100 for r in downstream_results]
f1_scores= [r['f1'] * 100 for r in downstream_results]

x    = np.arange(len(names))
w    = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - w/2, accs,      w, label='Accuracy (%)', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + w/2, f1_scores, w, label='F1-Score (%)', color='coral',     edgecolor='black')
ax.set_title('Downstream Classification — Raw vs Latent Features\n(Logistic Regression on MNIST)',
             fontweight='bold', fontsize=12)
ax.set_ylabel('Score (%)')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=15, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim([0, 105])
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

---
## Additional Task: Latent Space Sparsity Analysis

In [ ]:
# ================================================================
# ADDITIONAL: SPARSITY ANALYSIS
# ================================================================
# Compare how sparse the latent activations are across AE, DAE, SAE
# SAE should have the most zeros (most neurons inactive)

def sparsity_ratio(latent_vectors, threshold=0.1):
    """What fraction of latent neurons are close to 0?"""
    return (np.abs(latent_vectors) < threshold).mean()

ae_sparsity  = sparsity_ratio(latent_test_ae)
dae_sparsity = sparsity_ratio(latent_test_dae)
sae_sparsity = sparsity_ratio(latent_test_sae)

print("===== Sparsity Analysis (fraction of latent neurons near 0) =====")
print(f"  Basic AE   : {ae_sparsity*100:.1f}% of neurons inactive")
print(f"  Denoising AE: {dae_sparsity*100:.1f}% of neurons inactive")
print(f"  Sparse AE  : {sae_sparsity*100:.1f}% of neurons inactive  ← should be highest")

# Plot average activation per latent neuron for each model
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Average Latent Neuron Activation\n(Lower = More Sparse)',
             fontsize=12, fontweight='bold')

for ax, lat, title in [
    (axes[0], latent_test_ae,  'Basic AE'),
    (axes[1], latent_test_dae, 'Denoising AE'),
    (axes[2], latent_test_sae, 'Sparse AE'),
]:
    avg_activation = lat.mean(axis=0)  # average across all test images
    ax.bar(range(len(avg_activation)), avg_activation, color='steelblue', edgecolor='none')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Latent Neuron Index')
    ax.set_ylabel('Mean Activation')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=0.8)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print("SAE should show many bars close to zero — that's sparsity in action!")

---
## Final Summary

In [ ]:
# ================================================================
# FINAL SUMMARY
# ================================================================

print("=" * 68)
print("   AIML Lab Week 9 — Autoencoder Feature Learning — Summary")
print("=" * 68)
print()
print("DATASETS USED:")
print("  Image   : MNIST (28×28 grayscale handwritten digits, 70,000 images)")
print("  Tabular : Wine  (13 chemical features, 178 samples, 3 classes)")
print()
print("AUTOENCODERS BUILT:")
print(f"  1. Basic AE     — 784 → 256 → 128 → 32 (latent) → ... → 784")
print(f"     Test MSE : {ae_mse:.6f}")
print(f"  2. Denoising AE — same arch, trained on noisy→clean pairs")
print(f"     Test MSE : {dae_mse:.6f}")
print(f"  3. Sparse AE    — same arch + L1 penalty on latent layer")
print(f"     Test MSE : {sae_mse:.6f}")
print()
print("LATENT DIM TUNING:")
for r in latent_results:
    print(f"  latent={r['latent_dim']:>3} → MSE={r['test_mse']:.5f} ({r['compression']:.1f}x compression)")
print()
print("DOWNSTREAM CLASSIFICATION (Logistic Regression on MNIST):")
for r in downstream_results:
    print(f"  {r['name']:<25} → Accuracy={r['accuracy']*100:.2f}%  F1={r['f1']*100:.2f}%")
print()
print("KEY CONCLUSIONS:")
print("  1. Autoencoders learn compact representations WITHOUT labeled data")
print("  2. Denoising AE is more robust — learns structure, not noise")
print("  3. Sparse AE forces most latent neurons to be inactive")
print("  4. Larger latent dim = lower MSE but less compression")
print("  5. Latent features can match or exceed raw feature classification")
print("  6. t-SNE shows digit clusters in latent space — model learned digit structure!")